In [1]:
import pandas as pd
import duckdb
import re
from itertools import combinations
from optbinning import OptimalBinning
from joblib import Parallel, delayed
import multiprocessing
from typing import Dict, List, Optional

class StrategicSegmentBuilder:
    """
    A combinatorial heuristic engine for extracting highly predictive, mutually exclusive 
    segments from tabular data using Optimal Binning and Apriori-style pruning.

    Attributes:
        target (str): The dependent binary variable (e.g., 'default_flag').
        n_jobs (int): Number of CPU cores to utilize for parallel processing.
        min_sample_size (int): Minimum volume required for a segment to be valid.
        min_lift (float): Minimum lift (segment rate / base rate) required.
        top_n_vars (int): Number of top Information Value (IV) features to consider per iteration.
        max_segments (int): The maximum number of mutually exclusive segments to extract.
        enable_diversity (bool): If True, restricts rules from using features from the same group.
        enable_1way (bool): Include 1-dimensional rules in the final output.
        enable_2way (bool): Include 2-dimensional rules in the final output.
        enable_3way (bool): Include 3-dimensional rules in the final output.
        feature_groups (dict): Mapping of analytical categories to specific columns 
            (e.g., {'liquidity': ['avg_bal_1m', 'avg_bal_3m'], 'delinquency': ['pay_0', 'pay_2']}).
    """

    def __init__(self, 
                 target: str, 
                 n_jobs: int = -1, 
                 min_sample_size: int = 1000, 
                 min_lift: float = 2.0, 
                 top_n_vars: int = 20, 
                 max_segments: int = 10,
                 enable_diversity: bool = False, 
                 enable_1way: bool = True, 
                 enable_2way: bool = True, 
                 enable_3way: bool = True, 
                 feature_groups: Optional[Dict[str, List[str]]] = None):
        
        self.target = target
        self.n_jobs = n_jobs if n_jobs != -1 else max(1, multiprocessing.cpu_count() - 1)
        self.min_sample_size = min_sample_size
        self.min_lift = min_lift
        self.top_n_vars = top_n_vars
        self.max_segments = max_segments
        self.segments = []
        self.enable_diversity = enable_diversity
        self.enable_1way = enable_1way
        self.enable_2way = enable_2way
        self.enable_3way = enable_3way
        self.feature_groups = feature_groups or {}


    def _validate_feature_groups(self, df: pd.DataFrame):
        """
        Validates that all columns provided in feature_groups exist in the dataset.
        Raises a ValueError if a mismatch is found to prevent downstream failures.
        """
        if not self.feature_groups:
            return

        # Get all columns from the dataset except the target
        all_df_cols = set(df.columns) - {self.target}
        
        # Flatten all variables defined in the groups dictionary
        all_grouped_vars = set()
        for group, vars_list in self.feature_groups.items():
            for var in vars_list:
                if var not in all_df_cols:
                    raise ValueError(f"Validation Error: Variable '{var}' defined in group '{group}' "
                                     f"does not exist in the provided DataFrame.")
                all_grouped_vars.add(var)
        
        print(f"Feature group validation successful. {len(all_grouped_vars)} variables validated.")

    def get_group(self, var: str) -> str:
        """Identifies the assigned business category for a given feature."""
        for group, vars_list in self.feature_groups.items():
            if var in vars_list: 
                return group
        return var # Fallback: treat the raw variable name as its own standalone group
    
    def is_diverse(self, combo: tuple) -> bool:
        """
        Validates if a combination of features spans distinct analytical categories.
        Prevents redundant logic in rules (e.g., using two balance metrics simultaneously).
        """
        if not self.enable_diversity:
            return True
        
        groups = [self.get_group(v) for v in combo]
        return len(groups) == len(set(groups))
    
    def compute_iv_ranking(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Calculates the Information Value (IV) for all predictors against the target.
        Used dynamically in each iteration to select the most relevant remaining features.
        """
        def _get_iv(col):
            try:
                dtype = "categorical" if str(df[col].dtype) in ["object", "category"] else "numerical"
                optb = OptimalBinning(name=col, dtype=dtype)
                optb.fit(df[col].values, df[self.target].values)
                iv = optb.binning_table.build().IV.iloc[-1]
                return {"variable": col, "iv": iv * 100}
            except Exception:
                return {"variable": col, "iv": 0}

        cols = [c for c in df.columns if c != self.target]
        results = Parallel(n_jobs=self.n_jobs)(delayed(_get_iv)(col) for col in cols)
        return pd.DataFrame(results).sort_values("iv", ascending=False).reset_index(drop=True)

    def create_binned_df(self, df: pd.DataFrame, variables: List[str]) -> pd.DataFrame:
        """Discretizes continuous variables into optimal categorical bins to enable groupby operations."""
        binned_df = pd.DataFrame(index=df.index)
        for col in variables:
            dtype = "categorical" if str(df[col].dtype) in ["object", "category"] else "numerical"
            optb = OptimalBinning(name=col, dtype=dtype)
            optb.fit(df[col].values, df[self.target].values)
            
            # Cast as pd.Categorical to ensure massive memory savings
            transformed_bins = optb.transform(df[col], metric="bins")
            binned_df[col] = pd.Categorical(transformed_bins)
            
        binned_df[self.target] = df[self.target].values
        return binned_df

    def _agg_combinations(self, binned_df: pd.DataFrame, combo_list: List[tuple], base_rate: float) -> pd.DataFrame:
        """Parallelized aggregation to test all active combinations for lift and volume thresholds."""
        def _process_combo(combo):
            # observed=True prevents Pandas from calculating Cartesian products of empty categorical bins
            summary = binned_df.groupby(list(combo), observed=True).agg(
                count=(self.target, "size"), 
                events=(self.target, "sum")
            ).reset_index()
            
            summary = summary[summary["count"] >= self.min_sample_size].copy()
            if summary.empty: 
                return None
            
            summary["rate"] = (summary["events"] / summary["count"]) * 100
            summary["lift"] = summary["rate"] / (base_rate * 100)
            
            summary = summary[summary["lift"] >= self.min_lift]
            if summary.empty: 
                return None
            
            # Construct the raw logic string
            rule_parts = [f"{c}=" + summary[c].astype(str) for c in combo]
            summary["rule"] = " & ".join(rule_parts)
            summary["combo_vars"] = [combo] * len(summary)
                
            return summary[["rule", "count", "rate", "lift", "combo_vars"]]

        results = Parallel(n_jobs=self.n_jobs)(delayed(_process_combo)(c) for c in combo_list)
        valid_results = [r for r in results if r is not None]
        return pd.concat(valid_results, ignore_index=True) if valid_results else pd.DataFrame()

    def parse_rule_to_sql(self, rule_str: str) -> str:
        """Translates the OptBinning interval syntax into a production-ready SQL WHERE clause."""
        parts = [p.strip() for p in rule_str.split("&")]
        sql_conditions = []
        
        for part in parts:
            col, interval = part.split("=", 1)
            col, interval = col.strip(), interval.strip()
            
            # Catch string/categorical lists
            if interval.startswith("[") and "'" in interval:
                items = re.findall(r"'(.*?)'", interval)
                formatted_items = ", ".join([f"'{i}'" for i in items])
                sql_conditions.append(f"{col} IN ({formatted_items})")
                continue
                
            # Catch null/missing tags
            if interval in ["Special", "Missing"]:
                sql_conditions.append(f"{col} IS NULL")
                continue
                
            # Process numerical intervals
            if interval.startswith("[") or interval.startswith("("):
                left_bracket, right_bracket = interval[0], interval[-1]
                lower_str, upper_str = [x.strip() for x in interval[1:-1].split(",", 1)]
                
                col_conds = []
                if lower_str.lower() != '-inf':
                    op = '>=' if left_bracket == '[' else '>'
                    col_conds.append(f"{col} {op} {lower_str}")
                if upper_str.lower() != 'inf':
                    op = '<=' if right_bracket == ']' else '<'
                    col_conds.append(f"{col} {op} {upper_str}")
                    
                if col_conds:
                    sql_conditions.append(" AND ".join(col_conds))
                    
        return " AND ".join(f"({cond})" if "AND" in cond else cond for cond in sql_conditions)

    def extract_segments(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Iteratively extracts the highest-lift segments. Filters the dataset after 
        each extraction to ensure 100% mutually exclusive rule generation.
        """
        self._validate_feature_groups(df)
        current_df = df.copy()
        
        for i in range(1, self.max_segments + 1):
            base_rate = current_df[self.target].mean()
            if base_rate == 0 or len(current_df) < self.min_sample_size:
                break
                
            print(f"\n--- Iteration {i} | Remaining Rows: {len(current_df)} | Base Rate: {base_rate*100:.2f}% ---")
            
            iv_ranking = self.compute_iv_ranking(current_df)
            top_vars = iv_ranking.head(self.top_n_vars)["variable"].tolist()
            binned_df = self.create_binned_df(current_df, top_vars)
            
            all_rules = []
            
            # ==========================================
            # APRIORI LEVEL 1 (Base Features)
            # ==========================================
            combos_1 = [(c,) for c in top_vars]
            res_1 = self._agg_combinations(binned_df, combos_1, base_rate)
            
            valid_1way_vars = set()
            if not res_1.empty:
                # Store valid 1-way variables to prune the search space for Level 2 & 3
                valid_1way_vars = {c[0] for c in res_1["combo_vars"]}
                if self.enable_1way:
                    all_rules.append(res_1)
            
            if not valid_1way_vars:
                print("No base variables met the criteria. Stopping iterative extraction.")
                break
            
            # ==========================================
            # APRIORI LEVEL 2 (Pairs)
            # ==========================================
            valid_2way_sets = set()
            if len(valid_1way_vars) >= 2 and (self.enable_2way or self.enable_3way):
                # Ensure selected pairs adhere to the diversity constraint
                combos_2 = [c for c in combinations(valid_1way_vars, 2) if self.is_diverse(c)]
                
                if combos_2:
                    res_2 = self._agg_combinations(binned_df, combos_2, base_rate)
                    if not res_2.empty:
                        valid_2way_sets = {frozenset(c) for c in res_2["combo_vars"]}
                        if self.enable_2way:
                            all_rules.append(res_2)
            
            # ==========================================
            # APRIORI LEVEL 3 (Triplets)
            # ==========================================
            if self.enable_3way and len(valid_1way_vars) >= 3 and valid_2way_sets:
                combos_3 = []
                for combo in combinations(valid_1way_vars, 3):
                    if not self.is_diverse(combo):
                        continue
                    
                    # Apriori Check: A 3-way rule is only viable if all its 2-way subsets were viable
                    pairs = [frozenset(p) for p in combinations(combo, 2)]
                    if all(p in valid_2way_sets for p in pairs):
                        combos_3.append(combo)
                        
                if combos_3:
                    res_3 = self._agg_combinations(binned_df, combos_3, base_rate)
                    if not res_3.empty:
                        all_rules.append(res_3)
            
            # ==========================================
            # RULE SELECTION & POPULATION FILTERING
            # ==========================================
            if not all_rules:
                print("No active rules passed the criteria in this iteration. Stopping.")
                break

            shortlisted = pd.concat(all_rules, ignore_index=True)
            shortlisted = shortlisted.sort_values(["lift", "rate", "count"], ascending=False).reset_index(drop=True)
            
            best_rule = shortlisted.loc[0, "rule"]
            best_sql = self.parse_rule_to_sql(best_rule)
            
            self.segments.append({
                "segment_id": i,
                "rule_string": best_rule,
                "sql_filter": best_sql,
                "count": shortlisted.loc[0, "count"],
                "rate": shortlisted.loc[0, "rate"],
                "lift": shortlisted.loc[0, "lift"]
            })
            
            print(f"Segment {i} Found: {best_sql}")
            
            # Execute DuckDB pushdown to isolate the remaining unsegmented population
            current_df = duckdb.query(f"SELECT * FROM current_df WHERE NOT ({best_sql})").df()
            
        return pd.DataFrame(self.segments).drop(columns=["combo_vars"], errors="ignore")

    def evaluate_final_coverage(self, original_df: pd.DataFrame) -> pd.DataFrame:
        """Compiles the sequential segment rules into a full CASE WHEN SQL query to map coverage."""
        if not self.segments:
            return pd.DataFrame() # Return empty df if no rules were found
            
        case_statements = [f"WHEN {seg['sql_filter']} THEN {seg['segment_id']}" for seg in self.segments]
        case_sql = "\n                ".join(case_statements)
        
        final_query = f"""
        SELECT 
            CASE 
                {case_sql}
                ELSE 0 
            END AS segment, 
            COUNT(*) AS total_count,
            SUM("{self.target}") AS target_events,
            (SUM("{self.target}") * 100.0 / COUNT(*)) AS response_rate
        FROM original_df
        GROUP BY 1
        ORDER BY segment
        """
        
        return duckdb.query(final_query).df()

    
# Example Feature Groupings Setup
business_groups = {
"delinquency_vars":["dpd_avg_3m", "dpd_avg_6m", "dpd_avg_12m","max_dpd_12m"],            
"transaction_vars":["txn_count_avg_3m", "txn_count_avg_6m", "txn_count_avg_12m"],
"spend_vars":["spend_avg_3m", "spend_avg_6m", "spend_avg_12m"]
}

# Initialize the builder
builder = StrategicSegmentBuilder(
    target="default_flag", 
    n_jobs=-1, 
    min_sample_size=1000, 
    min_lift=2.0, 
    top_n_vars=15, 
    max_segments=10,
    enable_diversity=True,
    enable_1way=False,  # Exclude 1-way rules from final output
    enable_2way=True, 
    enable_3way=True,
    feature_groups=business_groups
)

data = pd.read_csv("credit_card_default_ssb_dataset_50000.csv")

segments_df = builder.extract_segments(data)
final_eval = builder.evaluate_final_coverage(data)
print(final_eval)


Feature group validation successful. 10 variables validated.

--- Iteration 1 | Remaining Rows: 50000 | Base Rate: 7.28% ---


TypeError: sequence item 0: expected str instance, Series found

In [ ]:
print("--- FULL SEGMENT RULES ---\n")

for index, row in segments_df.iterrows():
    print(f"Segment ID: {row['segment_id']}")
    print(f"Raw Rule:   {row['rule_string']}")
    print(f"SQL Filter: {row['sql_filter']}")
    print("-" * 50)

--- FULL SEGMENT RULES ---

Segment ID: 1
Raw Rule:   bureau_score=(-inf, 582.50) & max_dpd_12m=[33.62, inf)
SQL Filter: bureau_score < 582.50 AND max_dpd_12m >= 33.62
--------------------------------------------------
Segment ID: 2
Raw Rule:   payment_ratio_avg_6m=(-inf, 0.47) & payment_ratio_avg_3m=(-inf, 0.30) & bureau_score=(-inf, 592.50)
SQL Filter: payment_ratio_avg_6m < 0.47 AND payment_ratio_avg_3m < 0.30 AND bureau_score < 592.50
--------------------------------------------------
Segment ID: 3
Raw Rule:   missed_payments_last_6m=[2.50, inf) & max_dpd_12m=[29.97, inf)
SQL Filter: missed_payments_last_6m >= 2.50 AND max_dpd_12m >= 29.97
--------------------------------------------------
Segment ID: 4
Raw Rule:   payment_ratio_avg_6m=(-inf, 0.50) & max_dpd_12m=[29.33, inf)
SQL Filter: payment_ratio_avg_6m < 0.50 AND max_dpd_12m >= 29.33
--------------------------------------------------
Segment ID: 5
Raw Rule:   payment_ratio_avg_6m=(-inf, 0.45) & payment_ratio_avg_3m=(-inf, 0.37

In [ ]:
final_eval

,segment,total_count,target_events,response_rate
0,0,39073,716.0,1.832467
1,1,1015,564.0,55.566502
2,2,1102,544.0,49.364791
3,3,1130,410.0,36.283186
4,4,1103,355.0,32.184950
5,5,1231,250.0,20.308692
6,6,1058,227.0,21.455577
7,7,1145,196.0,17.117904
8,8,1072,178.0,16.604478
9,9,1001,118.0,11.788212
